<a href="https://colab.research.google.com/github/Noors-lab/VigIQ-Vigilance-Intelligence-/blob/main/adding_the_rule_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import zipfile
import os
from google.colab import drive

drive.mount('/content/drive')

root = "/content/drive/MyDrive/shoplifting"
extract_to = "/content/shoplifting_data"
os.makedirs(extract_to, exist_ok=True)

for file in os.listdir(root):
    if file.endswith('.zip'):
        zip_path = os.path.join(root, file)
        folder_name = file.replace('.zip', '').replace('.', '_')
        out_path = os.path.join(extract_to, folder_name)
        if not os.path.exists(out_path):
            print(f"Extracting {file}...")
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(out_path)
            print(f"  Done → {out_path}")
        else:
            print(f"Already extracted: {file}")

print("\nAll ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already extracted: shoplifting.zip
Already extracted: shoplifting_1.zip
Already extracted: shoplifting.v1i.yolov11.zip
Already extracted: Shoplifting.v2i.yolov11.zip

All ready.


In [4]:
!pip install ultralytics
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import time

# ─── Load model ───
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import json

# ─── LSTM Model definition ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256,
                 num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.classifier(out).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load LSTM
lstm_model = ShopliftingLSTM().to(device)
lstm_model.load_state_dict(torch.load(
    '/content/drive/MyDrive/shoplifting/vigiq_best_v4.pth',
    map_location=device
))
lstm_model.eval()

# Load normalization stats
X_mean = np.load('/content/drive/MyDrive/shoplifting/X_mean_v4.npy')[0]
X_std  = np.load('/content/drive/MyDrive/shoplifting/X_std_v4.npy')[0]

# Load YOLO pose
yolo = YOLO("yolo11n-pose.pt")

print("Models loaded. ✓")

# ─── KEYPOINT INDICES ───
NOSE         = 0
LEFT_SHOULDER  = 5
RIGHT_SHOULDER = 6
LEFT_ELBOW   = 7
RIGHT_ELBOW  = 8
LEFT_WRIST   = 9
RIGHT_WRIST  = 10
LEFT_HIP     = 11
RIGHT_HIP    = 12

# ─── NORMALIZATION ───
def normalize_keypoints(kps):
    left_hip  = kps[LEFT_HIP]
    right_hip = kps[RIGHT_HIP]
    center = (left_hip + right_hip) / 2
    left_shoulder  = kps[LEFT_SHOULDER]
    right_shoulder = kps[RIGHT_SHOULDER]
    shoulder_center = (left_shoulder + right_shoulder) / 2
    scale = np.linalg.norm(shoulder_center - center)
    if scale < 1e-6:
        scale = 1.0
    return ((kps - center) / scale).flatten().tolist()

# ─── RULE LAYER ───
def apply_rules(track_history, kps_sequence):
    """
    Returns (alert: bool, reason: str, confidence: float)
    """
    if len(kps_sequence) < 10:
        return False, "insufficient frames", 0.0

    # ── Rule 1: LSTM confidence ──
    seq = np.array(kps_sequence[-50:], dtype=np.float32)
    while len(seq) < 50:
        seq = np.vstack([seq, np.zeros((1, 34))])
    seq = (seq - X_mean) / (X_std + 1e-8)
    tensor = torch.tensor(seq ,dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        logit = lstm_model(tensor)
        confidence = torch.sigmoid(logit).item()

    if confidence < 0.80:
        return False, f"lstm confidence too low ({confidence:.2f})", confidence

    # ── Rule 2: Wrist destination check ──
    # Check if wrist moves toward hip area not upward/outward
    recent_kps = np.array(kps_sequence[-15:])  # last 15 frames
    if len(recent_kps) >= 5:
        # Left wrist y movement (y increases downward = toward hip)
        left_wrist_y  = recent_kps[:, LEFT_WRIST*2+1]
        right_wrist_y = recent_kps[:, RIGHT_WRIST*2+1]
        left_wrist_x  = recent_kps[:, LEFT_WRIST*2]
        right_wrist_x = recent_kps[:, RIGHT_WRIST*2]

        # Wrist moving toward center body = concealment
        left_moving_in  = abs(left_wrist_x[-1]  - left_wrist_x[0])  > 0.1
        right_moving_in = abs(right_wrist_x[-1] - right_wrist_x[0]) > 0.1
        wrist_destination_check = left_moving_in or right_moving_in
    else:
        wrist_destination_check = False

    if not wrist_destination_check:
        return False, "wrist not moving toward body", confidence

    # ── Rule 3: Sustained motion check ──
    # Alert only if suspicious for 2+ seconds (at 10fps sampling = 20 frames)
    sustained = len(kps_sequence) >= 20
    if not sustained:
        return False, "motion not sustained", confidence

    # ── Rule 4: Head rotation check ──
    # Nose x position variance = looking around
    nose_x_positions = np.array(kps_sequence[-20:])[:, NOSE*2]
    nose_variance = np.var(nose_x_positions)
    head_rotating = nose_variance > 0.01

    # ── Rule 5: Hooded person check ──
    # If nose keypoint confidence is low = face obscured = hooded
    # We approximate: if nose position is near zero = not detected = hooded
    nose_positions = np.array(kps_sequence[-10:])[:, NOSE*2]
    hooded = np.mean(np.abs(nose_positions)) < 0.05

    # ─── Final decision ───
    # Must pass Rule 1 + Rule 2 + Rule 3
    # Rule 4 and 5 are bonus signals that lower threshold
    base_rules_pass = confidence >= 0.80 and wrist_destination_check and sustained

    if hooded:
        # Hooded person — alert even with slightly lower confidence
        if confidence >= 0.65:
            return True, "HOODED PERSON + suspicious motion", confidence

    if base_rules_pass:
        if head_rotating:
            return True, "suspicious motion + head rotation", confidence
        else:
            return True, "suspicious motion detected", confidence

    return False, "rules not satisfied", confidence

print("Rule layer ready. ✓")

# ─── TEST ON VIDEO ───
def run_on_video(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    track_histories = defaultdict(list)
    kps_sequences   = defaultdict(list)
    alerted_ids     = set()

    frame_idx = 0
    alert_count = 0

    print(f"\nRunning on: {video_path}")
    print(f"FPS: {fps:.1f}\n")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % 3 == 0:  # sample every 3rd frame
            results = yolo.track(frame, persist=True, verbose=False)

            if results[0].boxes.id is not None:
                track_ids = results[0].boxes.id.cpu().numpy().astype(int)
                keypoints = results[0].keypoints.xy.cpu().numpy()

                for tid, kps in zip(track_ids, keypoints):
                    if kps.shape[0] == 17:
                        normalized = normalize_keypoints(kps)
                        kps_sequences[tid].append(normalized)

                        # Keep last 50 frames only
                        if len(kps_sequences[tid]) > 50:
                            kps_sequences[tid].pop(0)

                        # Don't re-alert same person
                        if tid in alerted_ids:
                            continue

                        # Apply rules
                        alert, reason, conf = apply_rules(
                            track_histories[tid],
                            kps_sequences[tid]
                        )

                        if alert:
                            timestamp = frame_idx / fps
                            print(f"🚨 ALERT — Person {tid}")
                            print(f"   Reason: {reason}")
                            print(f"   Confidence: {conf:.2f}")
                            print(f"   Timestamp: {timestamp:.1f}s")
                            print()
                            alerted_ids.add(tid)
                            alert_count += 1

        frame_idx += 1

    cap.release()
    print(f"Done. Total alerts: {alert_count}")

# ─── RUN ON A TEST CLIP ───
# Change this path to any video clip from your dataset
# Test more shoplifting clips
test_videos = [
    ("/content/shoplifting_data/shoplifting/shoplifting/shoplifting-19.mp4", "SHOPLIFTING"),
    ("/content/shoplifting_data/shoplifting/shoplifting/shoplifting-13.mp4", "SHOPLIFTING"),
    ("/content/shoplifting_data/shoplifting/normal/normal-75.mp4", "NORMAL"),
    ("/content/shoplifting_data/shoplifting/normal/normal-36.mp4", "NORMAL"),
]

for path, expected in test_videos:
    print(f"\nExpected: {expected}")
    run_on_video(path)
    print("─"*40)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Models loaded. ✓
Rule layer ready. ✓

Expected: SHOPLIFTING

Running on: /content/shoplifting_data/shoplifting/shoplifting/shoplifting-19.mp4
FPS: 30.0

Done. Total alerts: 0
────────────────────────────────────────

Expected: SHOPLIFTING

Running on: /content/shoplifting_data/shoplifting/shoplifting/shoplifting-13.mp4
FPS: 30.0

🚨 ALERT — Person 3
   Reason: suspicious motion detected
   Confidence: 0.88
   Timestamp: 3.6s

Done. Total alerts: 1
────────────────────────────────────────

Expected: NORMAL

Running on: /content/shoplifting_data/shoplifting/normal/normal-75.mp4
FPS: 30.0

Done. Total alerts: 0
────────────────────────────────────────

Expected: NORMAL

Running on: /content/shoplifting_data/shoplifting/normal/normal-36.mp4
FPS: 30.0

Done. Total alerts: 0
────────────────────────────────────────


In [5]:
import json
from google.colab import drive

drive.mount('/content/drive')

config = {
    "model": "vigiq_best_v4.pth",
    "lstm_threshold": 0.80,
    "hooded_threshold": 0.65,
    "min_frames": 20,
    "wrist_movement_threshold": 0.1,
    "nose_variance_threshold": 0.01,
    "sample_every": 3,
    "max_frames": 50,
    "version": "v4"
}

with open('/content/drive/MyDrive/shoplifting/vigiq_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Config saved to Drive ✓")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config saved to Drive ✓
